In [22]:
import numpy as np
import tensorflow as tf
import keras_tuner as kt
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras import layers, models, optimizers

In [26]:
# --- CONFIGURAZIONE ---
SEED = 123
tf.keras.utils.set_random_seed(SEED)

IMG_WIDTH, IMG_HEIGHT, IMG_CHANNELS = 224, 224, 3  #immagini RGB di dimensione 224x224
IMG_SHAPE = (IMG_WIDTH, IMG_HEIGHT, IMG_CHANNELS)

NUM_CLASSI = 5 # classifico 5 classi: Grano, Mais, Soia, Riso, Terreno vuoto
NUM_CAMPIONI = 200 # numero totale di immagini 
BATCH_SIZE = 8  #metto batch size piccolo per evitare problemi di memoria (ResNet50 è un modello pesante)
EPOCHS = 20  #numero di epoche per l'addestramento finale, da ottimizzare con il tuner

CLASS_NAMES = ["Grano", "Mais", "Soia", "Riso", "Terreno vuoto"]

AUTOTUNE = tf.data.AUTOTUNE

In [27]:
def genera_dati_agricoli(num_campioni, img_shape, num_classi,seed=SEED):
    """
    Simula il dataset 
    Le immagini sono randomiche e le etichette sono one-hot encoded.
    Restituisce X (immagini RGB) e y (etichette one-hot).
    """
    print(f"[DATI] Generazione di {num_campioni} campioni simulati (Noise)...")
    rng = np.random.default_rng(seed)
    
    # X: Immagini simulate (valori random 0-1)
    # Nota: ResNet di solito gradisce preprocessing specifico, ma per la struttura del codice
    # i valori normalizzati 0-1 funzionano tecnicamente per il training flow.
    X = rng.integers(low=0,high=256,size=(num_campioni,*img_shape)).astype(np.float32)

    # Classi simulate: numeri da 0 a 4
    y_classi = rng.integers(low=0,high=num_classi,size=(num_campioni,))
    #print(y_classi)  # Mostra le etichette simulate
    # Conversione in one-hot encoding
    y = np.eye(num_classi,dtype=np.float32)[y_classi]
    
    return X, y

# Generazione dei dati
X, y = genera_dati_agricoli(NUM_CAMPIONI, IMG_SHAPE, NUM_CLASSI,SEED)
print(f"Shape Input: {X.shape}, Shape Labels: {y.shape}")
#print(X[:5])  # Mostra le prime 5 immagini simulate
#print(y[:5])  # Mostra le prime 5 etichette one-hot

[DATI] Generazione di 200 campioni simulati (Noise)...
Shape Input: (200, 224, 224, 3), Shape Labels: (200, 5)


In [28]:
def split_dati(X, y):
    """
    Divido i dati in training, validation e test set.:
    
    70% training (140 campioni),  per aggiornare i pesi del modello
    15% validation (30 campioni), per controllare il modello durante l'addestramento 
    15% test (30 campioni), per valutare le prestazioni finali del modello (su dati mai usati prima)

    Stratifico in base alle classi per mantenere la distribuzione delle etichette.
    """
    y_classi = y.argmax(axis=1)

    X_train, X_temp, y_train, y_temp = train_test_split(
        X,
        y,
        test_size=0.30,
        random_state=SEED,
        stratify=y_classi
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp,
        y_temp,
        test_size=0.50,
        random_state=SEED,
        stratify=y_temp.argmax(axis=1)
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

X_train, X_val, X_test, y_train, y_val, y_test = split_dati(X, y)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape, "y_val:  ", y_val.shape)
print("X_test: ", X_test.shape, "y_test: ", y_test.shape)

X_train: (140, 224, 224, 3) y_train: (140, 5)
X_val:   (30, 224, 224, 3) y_val:   (30, 5)
X_test:  (30, 224, 224, 3) y_test:  (30, 5)


In [29]:
#------------------------------
# CREAZIONE DATASET TENSORFLOW
#------------------------------

def crea_dataset(X, y, batch_size, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))

    if shuffle:
        ds = ds.shuffle(buffer_size=len(X), seed=SEED)

    ds = ds.batch(batch_size)  #crea batch di dimensione batch_size
    ds = ds.prefetch(AUTOTUNE) #ottimizza il caricamento dei dati in memoria per evitare colli di bottiglia durante l'addestramento

    return ds

train_ds = crea_dataset(X_train, y_train, BATCH_SIZE, shuffle=True)
val_ds = crea_dataset(X_val, y_val, BATCH_SIZE, shuffle=False)
test_ds = crea_dataset(X_test, y_test, BATCH_SIZE, shuffle=False)


In [31]:
def build_model(hp):
    """
    Costruisce il modello definendo lo spazio di ricerca. A differenza del Grid Search, 
    l'ottimizzazione Bayesiana impara dai tentativi precedenti per trovare il setup ottimale.
    """
    # Spazio di ricerca per Learning Rate (logaritmico per coprire diversi ordini di grandezza)
    hp_lr = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log')
    # Dropout per prevenire l'overfitting (regolarizzazione)
    hp_dropout = hp.Float('dropout', 0.2, 0.5, step=0.1)
    #prova come dropout 0.2, 0.3, 0.4, 0.5 (step=0.1)
    #il dropout spegne casualmente una parte di neuroni durante l'addestramento (training) per evitare l'overfitting
    
    base_model=ResNet50(
        weights="imagenet", #carico pesi già addestrati su ImageNet
        include_top=False, #tolgo la testa (ultimo layer) perchè voglio fare classificazione su 5 classi, non sulle 1000 di ImageNet
        input_shape=IMG_SHAPE
    )
    
    # Freeze iniziale: non aggiorniamo i pesi del backbone per non distruggere 
    # le feature già apprese (bordi, forme) durante il tuning iniziale.
    base_model.trainable = False #blocco i pesi di MobileNet, durante il training non vengono modificati, vengono modificati solo i pesi dei layer aggiunti dopo (GlobalAveragePooling2D, Dropout, Dense)
    
    #modello sequenziale: aggiungo layer uno dopo l'altro
    model = models.Sequential([
        layers.Input(shape=IMG_SHAPE),
        
        # Data Augmentation integrata nel grafo: avviene in tempo reale su GPU.
        # Rende il modello robusto a variazioni di angolazione e zoom.
        #LAYERS DI DATA AUGMENTATION (per la parte di vision)
        #per aumentare i dati
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),

        layers.Lambda(preprocess_input), #preprocessing specifico per ResNet50, normalizza i pixel in modo che siano compatibili con i pesi pre-addestrati
        
        #passo l'immagine dentro MobileNet, che estrae feature complesse (bordi, forme, texture) senza bisogno di addestrare da zero.
        base_model,
        # Riduce la dimensionalità da (7, 7, 960) a (960,) calcolando la media, 
        # riducendo drasticamente il numero di parametri rispetto a un layer Flatten.
        layers.GlobalAveragePooling2D(), #riduce la dimensionalità
        layers.Dropout(hp_dropout), #per ottimizzare
        layers.Dense(128, activation='relu'), #Layer denso con 128 neuroni e attivazione ReLU
        layers.Dropout(hp_dropout), #per ottimizzare
        layers.Dense(NUM_CLASSI, activation='softmax') # Softmax per classificazione multi-classe mutua esclusiva
        #5 neuroni in output perchè ho 5 classi di fiori, softmax per avere la probabilità di appartenenza a ciascuno 
    ])
    
    #compilazione modello: definisco ottimizzatore, loss e metriche da monitorare
    model.compile(
        optimizer=optimizers.Adam(learning_rate=hp_lr),
        loss='categorical_crossentropy', # Usata perché le label sono ione-hot encoded non interi semplici
        metrics=['accuracy']
    )
    #adam aggiorna i pesi del modello in base al gradiente della loss rispetto ai pesi
    #learning_rate=hp_lr usa il learning rate scelto dal tuner
    #sparse_categorical_crossentropy è la loss più adatta per classificazione multi-classe con label interi (0,1,2,3,4)
    #se invece le label fossero one-hot encoded (es. [1,0,0,0,0] per la classe 0) si userebbe categorical_crossentropy
    return model

#OTTIMIZZAZIONE IPER-PARAMETRI

#KERAS TUNER: strumento per ottimizzare automaticamente gli iperparametri del modello (learning rate, dropout, ecc.)
# Configurazione del Tuner: BayesianOptimization è più efficiente del RandomSearch
# poiché modella probabilisticamente la funzione obiettivo.
tuner = kt.BayesianOptimization(
    build_model,
    objective='val_accuracy',
    max_trials=5, #5 tentativi per parametri
    directory='tuning_logs',
    project_name='progetto 4_tuning',
    overwrite=True
)
#dico a Keras di costruire il modello usando build_model, cambiare learning rate e dropout, 
#e scegliere quello con migliore val_accuracy
print("--- Ricerca Bayesiana del miglior set di Iperparametri ---")
tuner.search(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True
        )
    ]
)
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0] #prende la combinazione migliore

print("Best LR:",best_hps.get("learning_rate"))
print("Best Dropout:",best_hps.get("dropout"))

Trial 3 Complete [00h 06m 23s]
val_accuracy: 0.2666666805744171

Best val_accuracy So Far: 0.3333333432674408
Total elapsed time: 00h 18m 34s

Search: Running Trial #4

Value             |Best Value So Far |Hyperparameter
0.00014026        |0.00057648        |learning_rate
0.2               |0.3               |dropout

Epoch 1/20


KeyboardInterrupt: 

In [ ]:
# -----------------------------
# RICOSTRUZIONE MODELLO VINCENTE
# -----------------------------

# --- CALLBACKS E ADDESTRAMENTO FINALE ---

# Ricostruiamo il modello con i parametri vincenti trovati dal tuner
model = tuner.hypermodel.build(best_hps)
# salvo il modello migliore sulla validation accuracy, interrompo l'addestramento se la loss non scende più, e riduco il learning rate se il modello smette di imparare.
callbacks = [
    # Salvataggio nel nuovo formato .keras: più sicuro e leggero del vecchio .h5
    tf.keras.callbacks.ModelCheckpoint(
        'best_model_progetto4.keras', monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    # EarlyStopping: interrompe l'addestramento se la loss non scende più, 
    # evitando spreco di risorse e overfitting.
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
    ),
    # ReduceLROnPlateau: se il modello smette di imparare, "rallenta" per esplorare 
    # con più precisione i minimi della funzione di perdita.
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1 #se la val_loss non migliora per 5 epoche, dimezza il learning rate, ma non scendere sotto 1e-7
    )
]

In [ ]:
# -----------------------------
# ADDESTRAMENTO finale
# -----------------------------

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2011474123.py, line 8)

In [18]:
# -----------------------------
# VALUTAZIONE FINALE SU TEST SET
# -----------------------------

test_loss, test_accuracy = model.evaluate(test_ds)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 777ms/step - accuracy: 0.1667 - loss: 1.8319
Test Loss: 1.8319
Test Accuracy: 0.1667


Risultato scarso ma corretto per dati casuali.
Ottengo Accuracy 16.67%
con 30 immagini solo 5 saranno (in media) correte.